In [0]:
# Demo 14 Setup: Adding and Configuring a Genie Space
# Creates Gold-layer tables that students will connect to a Genie Space.

import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "main"
schema = f"demo_genie_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Schema: {catalog}.{schema}")

In [0]:
# --- gold_sales: 2000 rows ---
# A pre-joined Gold table with clean column names and pre-calculated fields.
# This is the kind of table you'd connect a Genie Space to.

random.seed(42)

spark.sql("DROP TABLE IF EXISTS gold_sales")

regions = ['Northeast', 'Southeast', 'Midwest', 'West', 'Northwest']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Food & Beverage']
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
channels = ['Online', 'Retail Store', 'Partner', 'Mobile App']
products = [
    ('Laptop Pro', 'Electronics'), ('Tablet Air', 'Electronics'), ('Wireless Earbuds', 'Electronics'),
    ('Smart Watch', 'Electronics'), ('Denim Jacket', 'Clothing'), ('Running Shoes', 'Clothing'),
    ('Winter Coat', 'Clothing'), ('Standing Desk', 'Home & Garden'), ('LED Lamp', 'Home & Garden'),
    ('Yoga Mat', 'Sports'), ('Dumbell Set', 'Sports'), ('Protein Bars', 'Food & Beverage'),
    ('Coffee Beans', 'Food & Beverage'), ('Garden Hose', 'Home & Garden'), ('Tennis Racket', 'Sports')
]

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("customer_segment", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_pct", DoubleType(), False),
    StructField("net_revenue", DoubleType(), False),
    StructField("cost", DoubleType(), False)
])

rows = []
for i in range(1, 2001):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    product = random.choice(products)
    unit_price = round(random.uniform(15, 500), 2)
    qty = random.randint(1, 6)
    discount = round(random.choice([0.0, 0.0, 0.0, 0.05, 0.10, 0.15, 0.20]), 2)
    net_rev = round(qty * unit_price * (1 - discount), 2)
    cost_val = round(net_rev * random.uniform(0.4, 0.7), 2)
    rows.append((
        i,
        order_date,
        random.choice(regions),
        product[0],
        product[1],
        random.choice(segments),
        random.choice(channels),
        qty,
        unit_price,
        discount,
        net_rev,
        cost_val
    ))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("gold_sales")
print(f"Created gold_sales: {df.count()} rows")

In [0]:
# --- gold_customers: 200 rows ---
# Customer-level summary for Genie to answer customer-related questions.

spark.sql("DROP TABLE IF EXISTS gold_customers")

random.seed(99)

cust_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("region", StringType(), False),
    StructField("signup_date", DateType(), False),
    StructField("lifetime_orders", IntegerType(), False),
    StructField("lifetime_revenue", DoubleType(), False),
    StructField("last_order_date", DateType(), False)
])

first_names = ['James', 'Mary', 'Robert', 'Patricia', 'John', 'Jennifer', 'Michael', 'Linda', 'David', 'Sarah']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Wilson', 'Taylor']

cust_rows = []
for i in range(1, 201):
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    signup = date(2020, 1, 1) + timedelta(days=random.randint(0, 1460))
    orders = random.randint(1, 50)
    revenue = round(orders * random.uniform(50, 400), 2)
    last_order = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    cust_rows.append((
        i,
        f"{fname} {lname}",
        random.choice(segments),
        random.choice(regions),
        signup,
        orders,
        revenue,
        last_order
    ))

df_cust = spark.createDataFrame(cust_rows, schema=cust_schema)
df_cust.write.saveAsTable("gold_customers")
print(f"Created gold_customers: {df_cust.count()} rows")

In [0]:
# --- Add Unity Catalog comments ---
# These comments are automatically picked up by Genie Spaces,
# improving its understanding without manual annotation.

# Table comments
spark.sql("COMMENT ON TABLE gold_sales IS 'Gold-layer order data with pre-joined dimensions. Each row is one order line with net revenue calculated after discounts.'")
spark.sql("COMMENT ON TABLE gold_customers IS 'Customer-level summary with lifetime metrics. Use for customer segmentation and retention analysis.'")

# Column comments - gold_sales
spark.sql("ALTER TABLE gold_sales ALTER COLUMN order_id COMMENT 'Unique order identifier'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN order_date COMMENT 'Date the order was placed'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN region COMMENT 'Geographic sales region (Northeast, Southeast, Midwest, West, Northwest)'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN product_name COMMENT 'Name of the product sold'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN product_category COMMENT 'Product grouping (Electronics, Clothing, Home & Garden, Sports, Food & Beverage)'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN customer_segment COMMENT 'Business size classification (Enterprise, Mid-Market, SMB, Consumer)'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN channel COMMENT 'Sales channel (Online, Retail Store, Partner, Mobile App)'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN quantity COMMENT 'Number of units ordered'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN unit_price COMMENT 'Price per unit before discount'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN discount_pct COMMENT 'Discount percentage applied (0.0 to 0.20)'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN net_revenue COMMENT 'Total revenue after discount: quantity * unit_price * (1 - discount_pct)'")
spark.sql("ALTER TABLE gold_sales ALTER COLUMN cost COMMENT 'Cost of goods sold for this order'")

# Column comments - gold_customers
spark.sql("ALTER TABLE gold_customers ALTER COLUMN customer_id COMMENT 'Unique customer identifier'")
spark.sql("ALTER TABLE gold_customers ALTER COLUMN customer_name COMMENT 'Full name of the customer'")
spark.sql("ALTER TABLE gold_customers ALTER COLUMN segment COMMENT 'Business size classification (Enterprise, Mid-Market, SMB, Consumer)'")
spark.sql("ALTER TABLE gold_customers ALTER COLUMN region COMMENT 'Geographic region of the customer'")
spark.sql("ALTER TABLE gold_customers ALTER COLUMN signup_date COMMENT 'Date the customer first registered'")
spark.sql("ALTER TABLE gold_customers ALTER COLUMN lifetime_orders COMMENT 'Total number of orders placed by this customer'")
spark.sql("ALTER TABLE gold_customers ALTER COLUMN lifetime_revenue COMMENT 'Total revenue from this customer across all orders'")
spark.sql("ALTER TABLE gold_customers ALTER COLUMN last_order_date COMMENT 'Date of the customers most recent order'")

print("Added table and column comments (Unity Catalog metadata)")

In [0]:
# --- Summary ---
print("="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Tables created:")
print(f"  gold_sales     - 2000 rows (order-level sales data)")
print(f"    Columns: order_id, order_date, region, product_name, product_category,")
print(f"             customer_segment, channel, quantity, unit_price, discount_pct,")
print(f"             net_revenue, cost")
print(f"")
print(f"  gold_customers - 200 rows (customer-level metrics)")
print(f"    Columns: customer_id, customer_name, segment, region, signup_date,")
print(f"             lifetime_orders, lifetime_revenue, last_order_date")
print(f"")
print(f"Unity Catalog comments added to all tables and columns.")
print(f"(Genie automatically uses these for better question understanding)")
print(f"")
print(f"Full table paths for Genie Space:")
print(f"  {catalog}.{schema}.gold_sales")
print(f"  {catalog}.{schema}.gold_customers")